In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil
from pathlib import Path
root = Path('/content/drive/MyDrive')

for name, out in [('Drought_S1_Beed_A','S1'), ('Drought_S1_Beed_B','S1'),
                  ('Drought_S2_A','S2'),      ('Drought_S2_B','S2')]:
    dst = root/'BEED_DATA'/out
    dst.mkdir(parents=True, exist_ok=True)
    for f in (root/name).glob('*.tif'):
        shutil.copy(f, dst/f.name)

In [ ]:
import re, numpy as np, pandas as pd, rasterio
from pathlib import Path

root = Path('/content/drive/MyDrive/BEED_DATA')

def index_files(folder):
    out = {}
    for f in sorted(Path(folder).glob('*.tif')):
        m = re.search(r'(\d{4})_(\d{2})', f.name)
        out[pd.Period(f"{m[1]}-{m[2]}")] = f
    return out

s1_idx = index_files(root/'S1')
s2_idx = index_files(root/'S2')
months = pd.period_range("2020-01", "2025-12", freq="M")

print("S1 files:", len(s1_idx), "| S2 files:", len(s2_idx))
print("S2 missing months:", [str(m) for m in months if m not in s2_idx])

# Compare the grid of every S2 file with the first S1 file
with rasterio.open(next(iter(s1_idx.values()))) as a:
    s1_meta = (a.crs, a.width, a.height, a.transform)
    print("\nS1 grid:", a.crs, "| size", a.width, "x", a.height, "| pixel", a.res)

bad_grid, metas = [], set()
for m, f in s2_idx.items():
    with rasterio.open(f) as s:
        metas.add((s.count, s.dtypes, s.descriptions))
        if (s.crs, s.width, s.height, s.transform) != s1_meta:
            bad_grid.append((f.name, s.width, s.height, s.res))

print("\nS2 files whose grid differs from S1:", len(bad_grid))
print(bad_grid[:3])
print("\nS2 band layouts found (count, dtype, band names):")
for x in metas: print(x)

with rasterio.open(next(iter(s2_idx.values()))) as b:
    print("\nS2 nodata value:", b.nodata)

S1 files: 72 | S2 files: 70
S2 missing months: ['2024-07', '2025-07']

S1 grid: EPSG:4326 | size 7147 x 3341 | pixel (0.00026949458523585647, 0.00026949458523585647)

S2 files whose grid differs from S1: 4
[('S2_BEED_2022_09.tif', 7122, 3459, (0.00026949458523585647, 0.00026949458523585647)), ('S2_BEED_2022_10.tif', 7122, 3459, (0.00026949458523585647, 0.00026949458523585647)), ('S2_BEED_2022_11.tif', 7122, 3459, (0.00026949458523585647, 0.00026949458523585647))]

S2 band layouts found (count, dtype, band names):
(10, ('int16', 'int16', 'int16', 'int16', 'int16', 'int16', 'int16', 'int16', 'int16', 'int16'), ('B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'NDMI', 'NBR'))

S2 nodata value: None


In [ ]:
check = ["2021-12", "2022-01", "2022-02", "2022-12", "2023-01"]
for c in check:
    m = pd.Period(c)
    if m not in s2_idx:
        print(c, "no file"); continue
    with rasterio.open(s2_idx[m]) as s:
        arr = s.read().astype("float32")
        if s.nodata is not None:
            arr[arr == s.nodata] = np.nan
    arr[arr == 0] = np.nan
    meds = np.nanmedian(arr.reshape(arr.shape[0], -1), axis=1)
    print(c, np.round(meds, 3))

2021-12 [  434.   671.   689.  2128.  1843.  1175.  4900. -5133.   716.  2663.]
2022-01 [  504.   755.   838.  2044.  1906.  1305.  3828. -4493.   261.  1885.]
2022-02 [  665.   888.  1056.  2080.  2083.  1487.  2942. -3868.  -133.  1374.]
2022-12 [  416.   659.   694.  2183.  1845.  1195.  4723. -5195.   584.  2508.]
2023-01 [  438.   695.   770.  2202.  1906.  1272.  4257. -4992.   360.  2146.]


##Fix the grid and check how missing pixels are stored

In [ ]:
import numpy as np, rasterio

with rasterio.open(s1_idx[months[0]]) as a:
    s1_b, px = a.bounds, a.res[0]
    print("S1 bounds:", s1_b)

diff_files = []
for m, f in s2_idx.items():
    with rasterio.open(f) as s:
        if (s.crs, s.width, s.height, s.transform) != s1_meta:
            diff_files.append(m)
            dx = (s.bounds.left - s1_b.left) / px
            dy = (s1_b.top - s.bounds.top) / px
            print(f.name, s.width, "x", s.height, s.bounds,
                  f"| offset from S1 in pixels: x={dx:.3f}, y={dy:.3f}")
print("Different files:", [str(m) for m in diff_files])

S1 bounds: BoundingBox(left=74.80819343186569, bottom=18.542035947982633, right=76.73427123254636, top=19.44241735725563)
S2_BEED_2022_09.tif 7122 x 3459 BoundingBox(left=74.816547764008, bottom=18.508618619413387, right=76.73588820005777, top=19.440800389744215) | offset from S1 in pixels: x=31.000, y=6.000
S2_BEED_2022_10.tif 7122 x 3459 BoundingBox(left=74.816547764008, bottom=18.508618619413387, right=76.73588820005777, top=19.440800389744215) | offset from S1 in pixels: x=31.000, y=6.000
S2_BEED_2022_11.tif 7122 x 3459 BoundingBox(left=74.816547764008, bottom=18.508618619413387, right=76.73588820005777, top=19.440800389744215) | offset from S1 in pixels: x=31.000, y=6.000
S2_BEED_2022_12.tif 7122 x 3459 BoundingBox(left=74.816547764008, bottom=18.508618619413387, right=76.73588820005777, top=19.440800389744215) | offset from S1 in pixels: x=31.000, y=6.000
Different files: ['2022-09', '2022-10', '2022-11', '2022-12']


In [ ]:
with rasterio.open(s2_idx[pd.Period("2022-07")]) as s:
    arr = s.read()
print("fraction of pixels exactly 0 in every band:", np.mean((arr == 0).all(axis=0)))
print("any -9999?:", (arr == -9999).any())
print("min / max:", arr.min(), arr.max())

fraction of pixels exactly 0 in every band: 0.47466068004412576
any -9999?: True
min / max: -10000 13252


## Fix the 4 files

This writes corrected copies into a new folder, so your originals are untouched:

In [ ]:
from rasterio.warp import reproject, Resampling

fix_dir = root/'S2_fixed'
fix_dir.mkdir(exist_ok=True)

with rasterio.open(s1_idx[months[0]]) as ref:
    dst_transform, dst_crs = ref.transform, ref.crs
    H, W = ref.height, ref.width

for m in diff_files:
    with rasterio.open(s2_idx[m]) as src:
        out = np.zeros((src.count, H, W), dtype="int16")   # 0 = missing (same as other files)
        for b in range(1, src.count + 1):
            reproject(rasterio.band(src, b), out[b - 1],
                      src_transform=src.transform, src_crs=src.crs,
                      dst_transform=dst_transform, dst_crs=dst_crs,
                      resampling=Resampling.nearest)
        profile = src.profile.copy()
        profile.update(width=W, height=H, transform=dst_transform)
        with rasterio.open(fix_dir / s2_idx[m].name, "w", **profile) as dst:
            dst.write(out)
            dst.descriptions = src.descriptions

# From now on, prefer the fixed files:
s2_idx.update(index_files(fix_dir))

In [ ]:
bad = []
for m, f in s2_idx.items():
    with rasterio.open(f) as s:
        if (s.crs, s.width, s.height, s.transform) != s1_meta:
            bad.append(str(m))
print("S2 files still off-grid:", bad)

S2 files still off-grid: []


In [ ]:
for m in diff_files:
    with rasterio.open(s2_idx[m]) as s:
        a = s.read(4)    # B8
    print(m, "fraction of S1 grid with S2 data:", round(float((a != 0).mean()), 3))

2022-09 fraction of S1 grid with S2 data: 0.569
2022-10 fraction of S1 grid with S2 data: 0.569
2022-11 fraction of S1 grid with S2 data: 0.569
2022-12 fraction of S1 grid with S2 data: 0.569


## one district mask, then clean Sentinel-2

The idea is simple. The district outline from Sentinel-1 is the truth for every source. Inside it, a Sentinel-2 value of 0 or -9999 means “no data”. Outside it, we ignore the pixel.

## build and save the district mask

In [ ]:
with rasterio.open(s1_idx[pd.Period("2022-07")]) as s:
    vv, vh = s.read(1), s.read(2)
district = ~((vv == 0) & (vh == 0))        # True = inside Beed
np.save(root/'district_mask.npy', district)
print("district share of grid:", round(float(district.mean()), 4))   # expect about 0.525

district share of grid: 0.5253


## Sentinel-2 loader (use this from now on)

In [ ]:
def load_s2(path):
    with rasterio.open(path) as s:
        a = s.read().astype("float32")                     # (10, H, W)
    bad = (a == -9999).any(axis=0) | ((a == 0).all(axis=0) & district)
    valid = district & ~bad                                # True = real observation
    a[:, ~valid] = np.nan
    a /= 10000.0                                           # undo the x10000 scaling
    a[:6] = np.clip(a[:6], 0, 1)                           # reflectance bands: cap bright outliers
    return a, valid

The first six bands are reflectance and the last four are indices. The valid layer is the validity channel the model will use.

## how much of the district has real Sentinel-2 data each month

In [12]:
rows = []
for m in months:
    if m not in s2_idx:
        rows.append((str(m), None)); continue
    with rasterio.open(s2_idx[m]) as s:
        b8 = s.read(4)
    valid = district & (b8 != -9999) & (b8 != 0)
    rows.append((str(m), round(float(valid.sum() / district.sum()), 3)))

for r in rows: print(r)
print("\nBelow 0.6:", [r[0] for r in rows if r[1] is not None and r[1] < 0.6])

('2020-01', 1.0)
('2020-02', 1.0)
('2020-03', 1.0)
('2020-04', 1.0)
('2020-05', 1.0)
('2020-06', 0.889)
('2020-07', 0.364)
('2020-08', 0.444)
('2020-09', 0.85)
('2020-10', 0.979)
('2020-11', 0.999)
('2020-12', 0.998)
('2021-01', 1.0)
('2021-02', 1.0)
('2021-03', 1.0)
('2021-04', 1.0)
('2021-05', 1.0)
('2021-06', 0.845)
('2021-07', 0.773)
('2021-08', 0.762)
('2021-09', 0.817)
('2021-10', 0.998)
('2021-11', 0.987)
('2021-12', 0.999)
('2022-01', 0.999)
('2022-02', 1.0)
('2022-03', 1.0)
('2022-04', 1.0)
('2022-05', 1.0)
('2022-06', 0.778)
('2022-07', 0.202)
('2022-08', 0.115)
('2022-09', 0.983)
('2022-10', 0.983)
('2022-11', 0.983)
('2022-12', 0.982)
('2023-01', 0.999)
('2023-02', 0.996)
('2023-03', 0.999)
('2023-04', 1.0)
('2023-05', 0.998)
('2023-06', 1.0)
('2023-07', 0.264)
('2023-08', 0.599)
('2023-09', 0.525)
('2023-10', 0.999)
('2023-11', 0.999)
('2023-12', 0.999)
('2024-01', 1.0)
('2024-02', 1.0)
('2024-03', 1.0)
('2024-04', 1.0)
('2024-05', 1.0)
('2024-06', 0.741)
('2024-07', None)

In [13]:
for c in ["2020-09", "2021-09", "2022-09", "2022-10", "2021-10"]:
    with rasterio.open(s2_idx[pd.Period(c)]) as s:
        b2 = s.read(1).astype("float32") / 10000     # blue band
    v = district & (b2 > 0)
    print(c, "median B2:", round(float(np.median(b2[v])), 4),
          "| share of pixels with B2 > 0.2 (bright, likely cloud):",
          round(float((b2[v] > 0.2).mean()), 4))

2020-09 median B2: 0.0603 | share of pixels with B2 > 0.2 (bright, likely cloud): 0.0166
2021-09 median B2: 0.0531 | share of pixels with B2 > 0.2 (bright, likely cloud): 0.0263
2022-09 median B2: 0.0618 | share of pixels with B2 > 0.2 (bright, likely cloud): 0.2048
2022-10 median B2: 0.0413 | share of pixels with B2 > 0.2 (bright, likely cloud): 0.0008
2021-10 median B2: 0.0438 | share of pixels with B2 > 0.2 (bright, likely cloud): 0.0002


In [14]:
with rasterio.open(s1_idx[pd.Period("2022-07")]) as s:
    vv1 = s.read(1)
analysis = district & (vv1 != -9999)       # inside Beed AND cropland
np.save(root/'analysis_mask.npy', analysis)
print("analysis share of grid:", round(float(analysis.mean()), 4))   # expect about 0.437

analysis share of grid: 0.4366


In [15]:
def load_s2(path):
    with rasterio.open(path) as s:
        a = s.read().astype("float32")
    bad = (a == -9999).any(axis=0) | (a == 0).all(axis=0)
    valid = analysis & ~bad                       # real observation inside cropland
    a[:, ~valid] = np.nan
    a /= 10000.0
    a[:6] = np.clip(a[:6], 0, 1)                  # cap bright reflectance outliers
    return a, valid

In [16]:
s1_partial = ["2020-02", "2022-09", "2022-10", "2023-03", "2023-12", "2024-06", "2024-07"]

recs = []
for m in months:
    if m not in s2_idx:
        recs.append(dict(month=str(m), s2_cov=0.0, s2_bright=np.nan))
        continue
    with rasterio.open(s2_idx[m]) as s:
        b2 = s.read(1).astype("float32") / 10000
        b8 = s.read(4)
    valid = analysis & (b8 != -9999) & (b8 != 0)
    recs.append(dict(month=str(m),
                     s2_cov=round(float(valid.sum() / analysis.sum()), 3),
                     s2_bright=round(float((b2[valid] > 0.2).mean()), 4) if valid.any() else np.nan))

flags = pd.DataFrame(recs)
flags["s1_partial"] = flags.month.isin(s1_partial)
flags["s2_absent"] = (flags.s2_cov < 0.3) | (flags.s2_bright > 0.10)
flags.to_csv(root/'month_flags.csv', index=False)

print(flags[flags.s2_absent | (flags.s2_cov < 0.6) | flags.s1_partial].to_string())
print("\nS2 absent months:", flags.month[flags.s2_absent].tolist())

      month  s2_cov  s2_bright  s1_partial  s2_absent
1   2020-02   1.000     0.0000        True      False
6   2020-07   0.364     0.0160       False      False
7   2020-08   0.452     0.0324       False      False
30  2022-07   0.197     0.0066       False       True
31  2022-08   0.118     0.0180       False       True
32  2022-09   0.985     0.2009        True       True
33  2022-10   0.985     0.0006        True      False
38  2023-03   1.000     0.0001        True      False
42  2023-07   0.270     0.0051       False       True
43  2023-08   0.598     0.0071       False      False
44  2023-09   0.527     0.0068       False      False
47  2023-12   1.000     0.0001        True      False
53  2024-06   0.749     0.0045        True      False
54  2024-07   0.000        NaN        True       True
55  2024-08   0.415     0.0220       False      False
66  2025-07   0.000        NaN       False       True
67  2025-08   0.111     0.0074       False       True

S2 absent months: ['2022-07

## clean weather table

In [10]:
months = pd.period_range("2020-01", "2025-12", freq="M")   # 72 monthly steps

In [11]:
import re, numpy as np, pandas as pd, rasterio
from pathlib import Path

root = Path('/content/drive/MyDrive/BEED_DATA')
months = pd.period_range("2020-01", "2025-12", freq="M")

def index_files(folder):
    out = {}
    for f in sorted(Path(folder).glob('*.tif')):
        m = re.search(r'(\d{4})_(\d{2})', f.name)
        out[pd.Period(f"{m[1]}-{m[2]}")] = f
    return out

s1_idx = index_files(root/'S1')
s2_idx = index_files(root/'S2')
s2_idx.update(index_files(root/'S2_fixed'))      # the four re-gridded months replace the originals

analysis = np.load(root/'analysis_mask.npy')
flags = pd.read_csv(root/'month_flags.csv').set_index('month')
print(len(s1_idx), len(s2_idx), analysis.shape, flags.shape)

72 70 (3341, 7147) (72, 4)


In [12]:
import re, numpy as np, pandas as pd, rasterio
from pathlib import Path

root = Path('/content/drive/MyDrive/BEED_DATA')

meteo = pd.read_excel(root/'Beed_Meteo_Clean_fusion_v7_finals.csv.xlsx')
meteo['month_id'] = pd.PeriodIndex(
    meteo.year.astype(str) + '-' + meteo.month.astype(str).str.zfill(2), freq='M')
meteo = meteo.set_index('month_id')
assert list(meteo.index) == list(months), "months don't line up"

weather_cols = ['smap_l4_sm_rootzone', 'chirps_rain_mm', 'rain_sum6_mm', 'water_balance_mm',
                'et_mm', 'tmax_c', 'tmin_c', 'et_pet_ratio', 'era5_sm_l1',
                'rain_anom_mm', 'lst_night_c', 'month_sin', 'month_cos']
label_cols = ['spi1_prov', 'spi3_prov', 'spi6_prov', 'drought_flag_spi3',
              'drought_flag_smap_rz', 'drought_flag_any', 'drought_any_next_month', 'cv_group']

meteo[weather_cols].to_csv(root/'weather_features.csv')
meteo[label_cols].to_csv(root/'labels.csv')
print(meteo[weather_cols].isna().sum().sum(), "missing values in weather features")

0 missing values in weather features


## build the dataset (satellite cube, weather and target together)

In [15]:
import re, time, warnings, numpy as np, pandas as pd, rasterio
from pathlib import Path

# ---------- setup ----------
root = Path('/content/drive/MyDrive/BEED_DATA')
months = pd.period_range("2020-01", "2025-12", freq="M")

def index_files(folder):
    out = {}
    for f in sorted(Path(folder).glob('*.tif')):
        m = re.search(r'(\d{4})_(\d{2})', f.name)
        out[pd.Period(f"{m[1]}-{m[2]}")] = f
    return out

s1_idx = index_files(root/'S1')
s2_idx = index_files(root/'S2')
s2_idx.update(index_files(root/'S2_fixed'))          # the 4 re-gridded months replace the originals
analysis = np.load(root/'analysis_mask.npy')
flags = pd.read_csv(root/'month_flags.csv').set_index('month')
print(len(s1_idx), len(s2_idx), analysis.shape, flags.shape)   # expect 72 70 (3341, 7147) (72, 4)

# ---------- 90 m block averaging ----------
BLOCK = 3
H = analysis.shape[0] // BLOCK * BLOCK
W = analysis.shape[1] // BLOCK * BLOCK
h90, w90 = H // BLOCK, W // BLOCK
cube_dir = root/'cube_monthly'; cube_dir.mkdir(exist_ok=True)

def block_mean(a, min_count=5):
    a = a[:H, :W].reshape(h90, BLOCK, w90, BLOCK)
    cnt = np.isfinite(a).sum(axis=(1, 3))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mean = np.nanmean(a, axis=(1, 3))
    mean[cnt < min_count] = np.nan
    return mean.astype("float32")

mask90 = block_mean(np.where(analysis, 1.0, np.nan).astype("float32"))
np.save(root/'mask90.npy', np.isfinite(mask90))

def s1_channels(path):
    with rasterio.open(path) as s:
        vv, vh = s.read(1), s.read(2)
    bad = (vv == -9999) | ((vv == 0) & (vh == 0)) | ~analysis
    vv = vv.astype("float32") / 100; vh = vh.astype("float32") / 100
    vv[bad | (vv > 5)] = np.nan
    vh[bad | (vh > 0)] = np.nan
    return [block_mean(vv), block_mean(vh), block_mean(vv - vh)]

def s2_channels(path):
    out = []
    with rasterio.open(path) as s:
        b8 = s.read(4)
        bad = (b8 == -9999) | (b8 == 0) | ~analysis
        for b in range(1, s.count + 1):
            a = s.read(b).astype("float32") / 10000
            if b <= 6: a = np.clip(a, 0, 1)
            a[bad] = np.nan
            out.append(block_mean(a))
    return out

# ---------- build ----------
TEST = False      # first run: builds ONE month so you can check speed and errors. Then set False.
todo = months[:1] if TEST else months

for m in todo:
    f = cube_dir / f"{m}.npy"
    if f.exists(): continue                      # resume after a disconnect
    t0 = time.time()
    ch = s1_channels(s1_idx[m])
    if (m in s2_idx) and not flags.loc[str(m), "s2_absent"]:
        ch += s2_channels(s2_idx[m])
    else:
        ch += [np.full((h90, w90), np.nan, "float32")] * 10
    np.save(f, np.stack(ch).astype("float16"))
    print("saved", m, f"{time.time() - t0:.0f}s")

72 70 (3341, 7147) (72, 4)
saved 2020-02 19s
saved 2020-03 21s
saved 2020-04 19s
saved 2020-05 21s
saved 2020-06 21s
saved 2020-07 17s
saved 2020-08 18s
saved 2020-09 19s
saved 2020-10 28s
saved 2020-11 23s
saved 2020-12 22s
saved 2021-01 25s
saved 2021-02 19s
saved 2021-03 21s
saved 2021-04 29s
saved 2021-05 21s
saved 2021-06 20s
saved 2021-07 19s
saved 2021-08 19s
saved 2021-09 20s
saved 2021-10 20s
saved 2021-11 20s
saved 2021-12 21s
saved 2022-01 21s
saved 2022-02 22s
saved 2022-03 20s
saved 2022-04 19s
saved 2022-05 20s
saved 2022-06 19s
saved 2022-07 4s
saved 2022-08 5s
saved 2022-09 4s
saved 2022-10 19s
saved 2022-11 21s
saved 2022-12 20s
saved 2023-01 24s
saved 2023-02 21s
saved 2023-03 21s
saved 2023-04 21s
saved 2023-05 22s
saved 2023-06 21s
saved 2023-07 6s
saved 2023-08 20s
saved 2023-09 19s
saved 2023-10 22s
saved 2023-11 21s
saved 2023-12 23s
saved 2024-01 22s
saved 2024-02 25s
saved 2024-03 26s
saved 2024-04 21s
saved 2024-05 22s
saved 2024-06 19s
saved 2024-07 5s
saved 

In [16]:
print(len(list(cube_dir.glob('*.npy'))), "monthly cube files (expect 72)")
a = np.load(cube_dir/"2022-07.npy").astype("float32")
print(a.shape)                                                  # expect (13, 1113, 2382)
print("S1 valid:", np.isfinite(a[0]).mean().round(3), "| S2 valid:", np.isfinite(a[3]).mean().round(3))

72 monthly cube files (expect 72)
(13, 1113, 2382)
S1 valid: 0.442 | S2 valid: 0.0


## Copy the cube to Colab’s local disk

In [17]:
!cp -r /content/drive/MyDrive/BEED_DATA/cube_monthly /content/cube_local
cube_dir = Path('/content/cube_local')
print(len(list(cube_dir.glob('*.npy'))), "files copied")

72 files copied


In [18]:
import numpy as np, pandas as pd, torch
from pathlib import Path
from torch.utils.data import Dataset

root = Path('/content/drive/MyDrive/BEED_DATA')
months = pd.period_range("2020-01", "2025-12", freq="M")

# use the local copy if you made one, otherwise read from Drive (slower)
cube_dir = Path('/content/cube_local') if Path('/content/cube_local').exists() else root/'cube_monthly'
print("cube folder:", cube_dir, "|", len(list(cube_dir.glob('*.npy'))), "files")

# ---- statistics from training years only (2020-2023) ----
train_months = [m for m in months if m.year <= 2023]
n_ch = 13
s = np.zeros(n_ch); ss = np.zeros(n_ch); n = np.zeros(n_ch)
for m in train_months:
    a = np.load(cube_dir/f"{m}.npy", mmap_mode='r')[:, ::4, ::4].astype('float32')
    f = np.isfinite(a); a0 = np.where(f, a, 0)
    s += a0.sum(axis=(1, 2)); ss += (a0**2).sum(axis=(1, 2)); n += f.sum(axis=(1, 2))
mean = (s / n).astype('float32')
std  = np.sqrt(ss / n - (s / n)**2).astype('float32') + 1e-6

weather = pd.read_csv(root/'weather_features.csv', index_col=0)
w_train = weather.loc[[str(m) for m in train_months]]
w_mean, w_std = w_train.mean().values, w_train.std().values + 1e-6
weather_vals = weather.values
spi = pd.read_csv(root/'labels.csv', index_col=0)['spi3_prov'].values
np.savez(root/'norm_stats.npz', mean=mean, std=std, w_mean=w_mean, w_std=w_std)

# ---- patch locations (64x64 at 90 m, at least half inside cropland) ----
mask90 = np.load(root/'mask90.npy')
P, STRIDE, T = 64, 32, 6
origins = [(r, c) for r in range(0, mask90.shape[0] - P + 1, STRIDE)
                  for c in range(0, mask90.shape[1] - P + 1, STRIDE)
           if mask90[r:r+P, c:c+P].mean() >= 0.5]
print(len(origins), "patches per month")

class BeedDataset(Dataset):
    def __init__(self, target_months, train=False, s2_dropout=0.2):
        self.t_idx = [months.get_loc(m) for m in target_months if months.get_loc(m) >= T - 1]
        self.train, self.p, self.cubes = train, s2_dropout, {}
    def _cube(self, i):
        if i not in self.cubes:
            self.cubes[i] = np.load(cube_dir/f"{months[i]}.npy", mmap_mode='r')
        return self.cubes[i]
    def __len__(self):
        return len(self.t_idx) * len(origins)
    def __getitem__(self, idx):
        t = self.t_idx[idx // len(origins)]
        r, c = origins[idx % len(origins)]
        x = np.stack([self._cube(i)[:, r:r+P, c:c+P].astype('float32')
                      for i in range(t - T + 1, t + 1)])            # (T, 13, P, P)
        x = (x - mean[None, :, None, None]) / std[None, :, None, None]
        v_s1 = np.isfinite(x[:, 0:1]); v_s2 = np.isfinite(x[:, 3:4])   # validity layers
        x = np.nan_to_num(x, nan=0.0)
        if self.train and np.random.rand() < self.p:                  # S2 dropout
            x[:, 3:] = 0; v_s2[:] = False
        x = np.concatenate([x, v_s1, v_s2], axis=1).astype('float32')  # (T, 15, P, P)
        w = ((weather_vals[t-T+1:t+1] - w_mean) / w_std).astype('float32')
        return torch.from_numpy(x), torch.from_numpy(w), torch.tensor(spi[t], dtype=torch.float32)

train_ds = BeedDataset([m for m in months if m.year <= 2023], train=True)
val_ds   = BeedDataset([m for m in months if m.year == 2024])
test_ds  = BeedDataset([m for m in months if m.year == 2025])
x, w, y = train_ds[0]
print(x.shape, w.shape, y)      # expect torch.Size([6, 15, 64, 64]) torch.Size([6, 13]) and an SPI value

cube folder: /content/cube_local | 72 files
1228 patches per month
torch.Size([6, 15, 64, 64]) torch.Size([6, 13]) tensor(1.7604)
